# 00 · Comprobación de instalación y contrato V8

No carga datos ni abre cohortes externas. Verifica dependencias,
versión, protocolo, archivos imprescindibles y sintaxis del código.


In [ ]:
from pathlib import Path
import ast, importlib, json, platform, sys

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "intergate").is_dir():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError("Ejecute dentro de InterGATE_V8.")
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

required_modules = {
    "numpy": "numpy", "pandas": "pandas", "scipy": "scipy",
    "scikit-learn": "sklearn", "statsmodels": "statsmodels",
    "torch": "torch", 
    "networkx": "networkx", "matplotlib": "matplotlib",
    "requests": "requests", "tqdm": "tqdm",
}
versions = {}
missing = []
for package, module_name in required_modules.items():
    try:
        module = importlib.import_module(module_name)
        versions[package] = getattr(module, "__version__", "available")
    except Exception as exc:
        missing.append(f"{package}: {exc}")
if missing:
    raise RuntimeError("Faltan dependencias:\n- " + "\n- ".join(missing))
print("Python:", platform.python_version())
print(json.dumps(versions, indent=2, ensure_ascii=False))


In [ ]:
import intergate
from intergate.config import (
    CFG, implementation_fingerprint, model_config_fingerprint,
)
from intergate.runtime_setup import assert_v8_protocol, assert_transport_protocol

assert intergate.__version__ == "8.1.0"
assert_v8_protocol(CFG)
assert_transport_protocol(CFG)
protocol = json.loads(
    (PROJECT_ROOT / "configs/protocol_v8.json").read_text(encoding="utf-8")
)
assert protocol["protocol_version"] == CFG.TRANSPORT_PROTOCOL_VERSION
samples_per_flow = CFG.GRAPH_BATCH_SIZE * CFG.TRANSPORT_STEPS_PER_EPISODE
optimizer_steps = (CFG.TRANSPORT_STEPS_PER_EPISODE + CFG.GRAPH_ACCUM_STEPS - 1) // CFG.GRAPH_ACCUM_STEPS
support_samples = CFG.GRAPH_BATCH_SIZE * CFG.TRANSPORT_SUPPORT_BATCHES
assert (samples_per_flow, optimizer_steps, support_samples) == (384, 2, 128)
print("InterGATE:", intergate.__version__)
print("Protocolo:", CFG.TRANSPORT_PROTOCOL_VERSION)
print(f"Presupuesto: {samples_per_flow} muestras/flujo, {optimizer_steps} updates/episodio, {support_samples} muestras/soporte")
print("Fingerprint:", model_config_fingerprint(CFG))
print("Implementación:", implementation_fingerprint())


In [ ]:
required_files = [
    PROJECT_ROOT / "README.md",
    PROJECT_ROOT / "RUN_FROM_SCRATCH.md",
    PROJECT_ROOT / "requirements.txt",
    PROJECT_ROOT / "configs/protocol_v8.json",
    PROJECT_ROOT / "resources_biological/PAM50_symbols.txt",
]
required_files += sorted((PROJECT_ROOT / "notebooks").glob("*.ipynb"))
absent = [str(path) for path in required_files if not path.is_file()]
if absent:
    raise FileNotFoundError("Faltan archivos:\n- " + "\n- ".join(absent))

syntax_errors = []
for path in sorted((PROJECT_ROOT / "intergate").glob("*.py")):
    try:
        ast.parse(path.read_text(encoding="utf-8"), filename=str(path))
    except SyntaxError as exc:
        syntax_errors.append(f"{path.name}: {exc}")
if syntax_errors:
    raise SyntaxError("\n".join(syntax_errors))
print(f"Código válido: {len(list((PROJECT_ROOT / 'intergate').glob('*.py')))} módulos.")
print("El test externo no se ha cargado.")
